# Mini pipeline — Celeb-DF (20 real + 20 fake videos)

**Goal:** check that every step works, from a video to a "fake" probability.
With only 40 videos the scores will be weak — **that is normal**. We are testing the pipeline, not the performance.

| Step | What it does |
|---|---|
| 0 | Install the libraries |
| 1 | Settings (folders, number of videos) |
| 2 | Find the Celeb-DF folder |
| 3 | Choose the videos and split train / validation / test **by person** |
| 4 | Test face cropping on ONE video |
| 5 | Crop faces from ALL chosen videos |
| 6 | Look at the cropped faces |
| 7 | Prepare the model (EfficientNet-B0) |
| 8 | Train |
| 9 | Evaluate on the test set |
| 10 | Predict one video → probability FAKE |

### How to use this notebook
- Click on a grey code cell, then press **Shift + Enter** to run it. Go **from top to bottom**, one cell at a time.
- Before each cell, read the box: it says what the cell does and **what you should see**.
- While a cell is running you see `[*]` on its left. Wait until it becomes a number like `[5]`.
- If something shows a red error: copy the error and send it to Claude.

## Step 0 — Install the libraries (only the first time)

Installs OpenCV (video + faces), PyTorch (the model), scikit-learn (metrics), pandas, matplotlib.

**You should see:** some lines of text, then it stops. It can take 1–3 minutes.
If it ends with a message asking to *restart the kernel*: menu **Kernel → Restart**, then continue at Step 1.

In [ ]:
%pip install -q "opencv-python>=4.8" torch torchvision scikit-learn pandas matplotlib pillow

## Step 1 — Settings

All the important numbers are here. For this mini test, **do not change anything**.

- `N_IDENTITIES = 20` → 20 people
- `VIDEOS_PER_ID = 1` → 1 real + 1 fake video per person → **20 real + 20 fake**
- `FRAMES_PER_VIDEO = 16` → 16 images taken from each video
- `IMAGE_SIZE = 224` → each face is resized to 224 × 224 (what EfficientNet-B0 expects)

**You should see:** the path of your project folder.

In [ ]:
from pathlib import Path
import random
import time

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project folder (this notebook lives in deepfake-detection-multimodal/notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

OUT_DIR = ROOT / "data" / "processed" / "celebdf_mini"   # where the cropped faces go
FACES_DIR = OUT_DIR / "faces"
CSV_PATH = OUT_DIR / "faces.csv"                          # list of all cropped faces
MODEL_PATH = ROOT / "models" / "video_efficientnet_b0_mini.pth"

N_IDENTITIES = 20       # number of people
VIDEOS_PER_ID = 1       # real videos AND fake videos per person
FRAMES_PER_VIDEO = 16   # images taken from each video
IMAGE_SIZE = 224        # size of each face crop
MARGIN = 0.3            # keep 30% extra around the face (hair, chin, edges of the fake)
SEED = 42               # same random choices every time you run

random.seed(SEED)
np.random.seed(SEED)
print("Project folder:", ROOT)

## Step 2 — Find the Celeb-DF folder

The cell looks for the folder that contains **`Celeb-real`** and **`Celeb-synthesis`**,
first in `data/raw/celebdf/`, then in your **Downloads** folder.

**You should see:** the folder path, then about **590 real** and **5639 fake** videos.

**If it says "not found":** in Finder, find the folder that contains `Celeb-real`, right-click it while holding the
**Option** key → **Copy "…" as Pathname**, then paste it in the line
`CELEBDF_DIR = Path("...")` below (remove the `#` at the start) and run the cell again.

In [ ]:
import os
import subprocess


def find_celebdf():
    blocked = []
    # 1) ask Spotlight (finds the folder anywhere on the Mac)
    try:
        out = subprocess.run(["mdfind", "kMDItemFSName == 'Celeb-real'"],
                             capture_output=True, text=True, timeout=30).stdout
        for line in out.splitlines():
            p = Path(line)
            if p.is_dir() and (p.parent / "Celeb-synthesis").is_dir():
                return p.parent, blocked
    except Exception:
        pass
    # 2) search the usual folders
    home = Path.home()
    for base in [ROOT / "data" / "raw" / "celebdf", home / "Downloads", home / "Desktop", home / "Documents"]:
        for dirpath, dirnames, _ in os.walk(base, onerror=lambda e: blocked.append(e.filename)):
            if "Celeb-real" in dirnames and "Celeb-synthesis" in dirnames:
                return Path(dirpath), blocked
            if len(Path(dirpath).relative_to(base).parts) >= 4:
                dirnames[:] = []
            else:
                dirnames[:] = [d for d in dirnames if not d.startswith(".")]
    return None, blocked


CELEBDF_DIR, blocked = find_celebdf()
# CELEBDF_DIR = Path("/Users/YOUR_NAME/Desktop/archive")   # <- only if not found automatically

if CELEBDF_DIR is None:
    print("Celeb-DF not found.")
    if blocked:
        print("The Mac blocked these folders:", blocked)
        print("-> System Settings > Privacy & Security > Files and Folders > Terminal: turn ON Downloads/Desktop/Documents")
else:
    real_videos = sorted((CELEBDF_DIR / "Celeb-real").glob("*.mp4"))
    fake_videos = sorted((CELEBDF_DIR / "Celeb-synthesis").glob("*.mp4"))
    print("Celeb-DF folder:", CELEBDF_DIR)
    print("Real videos:", len(real_videos))
    print("Fake videos:", len(fake_videos))

## Step 3 — Choose the videos and split them **by person**

Celeb-DF file names tell you who is in the video:
- real: `id3_0007.mp4` → person **id3**
- fake: `id3_id12_0007.mp4` → made from a video of person **id3**

**Why split by person?** A fake video is built from a real video of the same person (same room, same clothes).
If the real one is in *train* and the fake one in *test*, the model can "recognise the room" instead of
detecting the fake → results look better than they really are (**data leakage**).
So all videos of one person go into the **same** group:

- **train** (70%) → the model learns from these
- **validation** (15%) → we check the model after each training round
- **test** (15%) → used **only once at the end** (Step 9)

**You should see:** a small table with the number of real (0) and fake (1) videos in each group.

In [ ]:
def person_of(video_path):
    # "id3_0007" -> "id3"   and   "id3_id12_0007" -> "id3"
    return video_path.stem.split("_")[0]


real_by_person, fake_by_person = {}, {}
for v in real_videos:
    real_by_person.setdefault(person_of(v), []).append(v)
for v in fake_videos:
    fake_by_person.setdefault(person_of(v), []).append(v)

# keep only people who have BOTH real and fake videos
people = sorted(set(real_by_person) & set(fake_by_person))
chosen = random.sample(people, min(N_IDENTITIES, len(people)))

rows = []
for person in chosen:
    for v in random.sample(real_by_person[person], min(VIDEOS_PER_ID, len(real_by_person[person]))):
        rows.append({"video_path": v, "video": v.name, "label": 0, "person": person})
    for v in random.sample(fake_by_person[person], min(VIDEOS_PER_ID, len(fake_by_person[person]))):
        rows.append({"video_path": v, "video": v.name, "label": 1, "person": person})
videos = pd.DataFrame(rows)

# split the PEOPLE, then every video follows its person
random.shuffle(chosen)
n_test = max(2, round(0.15 * len(chosen)))
n_val = max(2, round(0.15 * len(chosen)))
split_of = {}
for i, person in enumerate(chosen):
    split_of[person] = "test" if i < n_test else "val" if i < n_test + n_val else "train"
videos["split"] = videos["person"].map(split_of)

print("People chosen:", len(chosen))
print(pd.crosstab(videos["split"], videos["label"], margins=True))

## Step 4 — Test face cropping on ONE video

For each video:
1. take **16 frames** spread evenly from start to end,
2. find the **face** in each frame (OpenCV's YuNet face detector — the first time, it downloads a small 230 KB file),
3. cut a square around the face with a small margin,
4. resize it to **224 × 224**.

**You should see:** up to 8 face images from one video. They should be **faces**, well centred.
If you see walls or hands instead of faces, tell Claude.

In [ ]:
import urllib.request

# YuNet = OpenCV's face detector (small file, downloaded once into models/)
DETECTOR_PATH = ROOT / "models" / "face_detection_yunet_2023mar.onnx"
DETECTOR_URL = ("https://media.githubusercontent.com/media/opencv/opencv_zoo/main/"
                "models/face_detection_yunet/face_detection_yunet_2023mar.onnx")
if not DETECTOR_PATH.exists() or DETECTOR_PATH.stat().st_size < 100_000:
    DETECTOR_PATH.parent.mkdir(parents=True, exist_ok=True)
    print("Downloading the face detector (230 KB)...")
    urllib.request.urlretrieve(DETECTOR_URL, DETECTOR_PATH)
FACE_DETECTOR = cv2.FaceDetectorYN.create(str(DETECTOR_PATH), "", (320, 320), 0.7, 0.3, 50)
DETECT_MAX_SIDE = 640   # frames are shrunk to this size to find the face faster


def crop_face(frame):
    h_img, w_img = frame.shape[:2]
    scale = min(1.0, DETECT_MAX_SIDE / max(h_img, w_img))
    small = cv2.resize(frame, (int(w_img * scale), int(h_img * scale))) if scale < 1 else frame
    FACE_DETECTOR.setInputSize((small.shape[1], small.shape[0]))
    _, faces = FACE_DETECTOR.detect(small)
    if faces is None or len(faces) == 0:
        return None
    x, y, w, h = (max(faces, key=lambda f: f[2] * f[3])[:4] / scale)   # biggest face, back to full size
    cx, cy = x + w / 2, y + h / 2
    side = max(w, h) * (1 + MARGIN)
    x0, y0 = int(max(0, cx - side / 2)), int(max(0, cy - side / 2))
    x1 = int(min(frame.shape[1], cx + side / 2))
    y1 = int(min(frame.shape[0], cy + side / 2))
    face = frame[y0:y1, x0:x1]
    return cv2.resize(face, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)


def extract_faces(video_path, n_frames=FRAMES_PER_VIDEO):
    """Returns a list of (frame_number, face_image) for one video."""
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    results = []
    for i in np.unique(np.linspace(0, total - 1, n_frames).astype(int)):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, frame = cap.read()
        if not ok:
            continue
        face = crop_face(frame)
        if face is not None:
            results.append((int(i), face))
    cap.release()
    return results


test_video = videos.iloc[0]["video_path"]
faces = extract_faces(test_video)
print(f"{test_video.name}: {len(faces)} faces found out of {FRAMES_PER_VIDEO} frames")

fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for ax in axes:
    ax.axis("off")
for ax, (frame_number, face) in zip(axes, faces[:8]):
    ax.imshow(cv2.cvtColor(face, cv2.COLOR_BGR2RGB))   # OpenCV uses BGR colours, matplotlib uses RGB
    ax.set_title(f"frame {frame_number}", fontsize=8)
plt.show()

## Step 5 — Crop faces from ALL chosen videos

Same as Step 4, for the 40 videos. Every face is saved as a `.jpg` in `data/processed/celebdf_mini/faces/`,
and a table **`faces.csv`** lists each image with its video, label (0 = real, 1 = fake), person and split.

**You should see:** one line per video, e.g. `[3/40] id5_0002.mp4: 16 faces`. It takes about 2–5 minutes.
At the end, a summary table.

In [ ]:
FACES_DIR.mkdir(parents=True, exist_ok=True)
records, no_face = [], []
start = time.time()

for k, row in enumerate(videos.itertuples(), start=1):
    faces = extract_faces(row.video_path)
    if not faces:
        no_face.append(row.video)
    for frame_number, face in faces:
        out = FACES_DIR / f"{row.video_path.stem}_f{frame_number:05d}.jpg"
        cv2.imwrite(str(out), face)
        records.append({
            "image": str(out.relative_to(ROOT)),
            "video": row.video,
            "label": row.label,
            "person": row.person,
            "split": row.split,
            "frame": frame_number,
        })
    print(f"[{k}/{len(videos)}] {row.video}: {len(faces)} faces")

faces_df = pd.DataFrame(records)
faces_df.to_csv(CSV_PATH, index=False)
print(f"\nDone in {time.time() - start:.0f} s — {len(faces_df)} face images saved")
print("Videos with no face found:", no_face if no_face else "none")
print(pd.crosstab(faces_df["split"], faces_df["label"], margins=True))

## Step 6 — Look at the cropped faces

Top row = **REAL**, bottom row = **FAKE**. Often you cannot see the difference with your eyes — that is
exactly why we train a model.

**You should see:** 2 rows of 6 faces.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for row_axes, label, name in [(axes[0], 0, "REAL"), (axes[1], 1, "FAKE")]:
    sample = faces_df[faces_df["label"] == label].sample(min(6, (faces_df["label"] == label).sum()), random_state=SEED)
    for ax in row_axes:
        ax.axis("off")
    for ax, image_path in zip(row_axes, sample["image"]):
        ax.imshow(plt.imread(ROOT / image_path))
        ax.set_title(name, fontsize=9, color="green" if label == 0 else "red")
plt.tight_layout()
plt.show()

## Step 7 — Prepare the model

- **EfficientNet-B0** already trained on ImageNet (millions of photos): it already "knows how to see".
- We replace its last layer by **1 output** = how fake the face looks. A *sigmoid* turns it into a probability between 0 and 1.
- **Data augmentation** on training images only (random flip, small colour changes) so the model does not memorise.

If you closed or restarted Jupyter after Step 5: run **Step 1, Step 2 and Step 4** again (fast), then continue
here — this cell reloads `faces.csv`, so you do **not** need to crop again.

**You should see:** the device (`mps` = your Mac's GPU) and the number of images in each group.

In [ ]:
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class FaceDataset(Dataset):
    def __init__(self, df, tf):
        self.df = df.reset_index(drop=True)
        self.tf = tf

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        image = Image.open(ROOT / row["image"]).convert("RGB")
        return self.tf(image), torch.tensor(float(row["label"]))


faces_df = pd.read_csv(CSV_PATH)
train_df = faces_df[faces_df["split"] == "train"]
val_df = faces_df[faces_df["split"] == "val"]
test_df = faces_df[faces_df["split"] == "test"]

train_loader = DataLoader(FaceDataset(train_df, train_tf), batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(FaceDataset(val_df, eval_tf), batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(FaceDataset(test_df, eval_tf), batch_size=32, shuffle=False, num_workers=0)

model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)   # 1 output: REAL (0) / FAKE (1)
model = model.to(device)

print("Device:", device)
print(f"Images -> train: {len(train_df)} | validation: {len(val_df)} | test: {len(test_df)}")

## Step 8 — Train

One **epoch** = the model sees every training image once. We do 3 epochs.
After each epoch we measure on the **validation** set and keep the best model (lowest validation loss)
in `models/video_efficientnet_b0_mini.pth`.

- **loss** = how wrong the model is → should go **down**
- **acc** = % of images classified correctly

**You should see:** 3 lines (one per epoch), then a graph of training and validation loss.
On your Mac it takes about 1–5 minutes. With so few videos, validation can jump around — normal.

In [ ]:
EPOCHS = 3
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)


def run_epoch(loader, training):
    model.train(training)
    total_loss, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images).squeeze(1)
            loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            correct += ((logits > 0).float() == labels).sum().item()
            count += len(labels)
    return total_loss / count, correct / count


MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
history, best_val_loss = [], float("inf")
for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss, val_acc = run_epoch(val_loader, training=False)
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
    saved = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_PATH)
        saved = "  <- best so far, saved"
    print(f"Epoch {epoch}/{EPOCHS} | train loss {train_loss:.3f} acc {train_acc:.0%} | "
          f"val loss {val_loss:.3f} acc {val_acc:.0%} | {time.time() - start:.0f} s{saved}")

h = pd.DataFrame(history)
plt.plot(h["epoch"], h["train_loss"], marker="o", label="train")
plt.plot(h["epoch"], h["val_loss"], marker="o", label="validation")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.title("Loss per epoch")
plt.show()

## Step 9 — Evaluate on the test set (only once!)

We load the best model and predict on the **test** people, which the model has **never seen**.

Two levels:
- **frame-level**: each face image is judged alone
- **video-level**: we **average** the probabilities of the 16 faces of a video → one score per video
  (this is what the final project uses)

Metrics: **Accuracy, Precision, Recall, F1-score, AUC**, plus a **confusion matrix**.

**You should see:** a table of metrics and a 2 × 2 confusion matrix.
With only ~6 test videos, the numbers are not meaningful yet — the goal is that it **runs**.

In [ ]:
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

probabilities = []
with torch.no_grad():
    for images, _ in test_loader:
        probabilities += torch.sigmoid(model(images.to(device)).squeeze(1)).cpu().tolist()

test_results = test_df.copy()
test_results["p_fake"] = probabilities
video_results = test_results.groupby(["video", "label"], as_index=False)["p_fake"].mean()


def metrics(y_true, p_fake):
    y_pred = (p_fake >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, p_fake) if len(set(y_true)) == 2 else float("nan"),
    }


report = pd.DataFrame([
    metrics(test_results["label"], test_results["p_fake"]),
    metrics(video_results["label"], video_results["p_fake"]),
], index=["frame-level", "video-level"]).round(3)
print(report)
print()
print(video_results.round(3))

cm = confusion_matrix(video_results["label"], (video_results["p_fake"] >= 0.5).astype(int), labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=["REAL", "FAKE"]).plot(cmap="Blues")
plt.title("Confusion matrix (video-level)")
plt.show()

## Step 10 — Predict one video → probability FAKE

This is the function the final project needs: **give a video, get a number between 0 and 1**.
Close to 1 = probably fake, close to 0 = probably real.

**You should see:** one test video, its true label, and the predicted probability.
You can also put the path of **any** `.mp4` in `predict_video(...)`.

In [ ]:
def predict_video(video_path):
    faces = extract_faces(video_path)
    if not faces:
        return None   # no face found
    batch = torch.stack([
        eval_tf(Image.fromarray(cv2.cvtColor(face, cv2.COLOR_BGR2RGB))) for _, face in faces
    ]).to(device)
    with torch.no_grad():
        return torch.sigmoid(model(batch).squeeze(1)).mean().item()


example = video_results.iloc[0]
folder = "Celeb-real" if example["label"] == 0 else "Celeb-synthesis"
p = predict_video(CELEBDF_DIR / folder / example["video"])
print(f"Video: {example['video']}")
print(f"True label: {'FAKE' if example['label'] == 1 else 'REAL'}")
print(f"Predicted probability FAKE: {p:.2f} -> {'FAKE' if p >= 0.5 else 'REAL'}")

## Done — the pipeline works from start to end

**What you just did:** video → frames → face crops → split by person → training → evaluation → prediction.

**Next (when you are ready for the real run):**
1. In Step 1, set `N_IDENTITIES = 50` and `VIDEOS_PER_ID = 4` → **200 real + 200 fake** videos.
2. Run all steps again. Cropping will take longer (about 20–40 minutes).
3. For more epochs (e.g. `EPOCHS = 10`), Google Colab with a GPU is faster.

Send Claude: the table from Step 9 and the loss graph from Step 8.